# 🎁 Bonus — See It & Predict It
### Superstore analyst, day one (continued)

You've cleaned the data and briefed leadership. Now the fun part: **see** your
data as charts, and ask a bigger question — *can Python learn to predict?*

> Run the cells in order. A couple have a small `# TODO` for you to complete.
> Needs `output/clean.csv`, so run `python src/clean.py` first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Find output/clean.csv whether this notebook runs from the repo root or notebooks/.
clean_path = next((p for p in [Path("output/clean.csv"), Path("../output/clean.csv")]
                   if p.exists()), None)
if clean_path is None:
    raise FileNotFoundError("output/clean.csv not found — run `python src/clean.py` first.")

df = pd.read_csv(clean_path, parse_dates=["order_date", "ship_date"])
print(df.shape)
df.head()

---
## Part A — See your data 📊

With clean data, a chart is **one line** of pandas. (Day 3 goes deep on this —
here's a taste.)

In [ ]:
# Total sales by region.
df.groupby("region")["sales"].sum().sort_values().plot(
    kind="barh", title="Total sales by region")

In [ ]:
# Sales trend across the months.
df.groupby(df["order_date"].dt.month)["sales"].sum().plot(
    marker="o", title="Sales by month")

In [ ]:
# Profit by category — any surprises?
df.groupby("category")["profit"].sum().plot(kind="bar", title="Profit by category")

In [ ]:
# 🏋️ Your turn: chart the total quantity sold per region as a bar chart.
# TODO: group by region, sum 'quantity', then .plot(kind="bar", title="Quantity by region")

---
## Part B — Your first ML model: predict profit 🤖

**Machine learning** = let the computer learn patterns from data to make
predictions. Let's ask: *given an order's sales, quantity and discount, can we
predict its profit?*

We split the data into a **training** set (the model learns from it) and a
**test** set (unseen orders, to check it generalises).

In [ ]:
# Models can't handle missing values. Keep only the rows that have all the
# numbers we need. (Your cleaned data should already be fine — this is a safety net.)
ml_df = df.dropna(subset=["sales", "quantity", "discount", "profit"]).copy()
print(f"{len(ml_df)} rows ready for modelling")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

features = ["sales", "quantity", "discount"]   # the "clues"
X = ml_df[features]
y = ml_df["profit"]                            # what we want to predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

model = LinearRegression()
# TODO: train the model on the training data
# hint: model.fit(X_train, y_train)

print("R² on unseen orders:", round(model.score(X_test, y_test), 3))

In [ ]:
# Use the trained model to predict the profit of a brand-new order.
new_order = pd.DataFrame({"sales": [500], "quantity": [4], "discount": [0.1]})
print("Predicted profit: €", round(model.predict(new_order)[0], 2))

---
## Part C — Classification: *will* an order be profitable? 🎯

Same idea, different question — a yes/no prediction instead of a number. We
label each order `1` (made a profit) or `0` (didn't), and train a model to guess.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

ml_df["profitable"] = (ml_df["profit"] > 0).astype(int)   # 1 = profit, 0 = loss

X = ml_df[["sales", "quantity", "discount"]]
y = ml_df["profitable"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

clf = DecisionTreeClassifier(max_depth=3, random_state=0)
# TODO: train the classifier
# hint: clf.fit(X_train, y_train)

print("Accuracy on unseen orders:", round(clf.score(X_test, y_test), 3))

---
## That's a wrap 🎉

In a few lines you charted real data **and** built two predictive models. This
is just a spark — predicting and modelling is a whole field (and a future
training!). For now: you cleaned messy data and made it *talk*.

💬 **Discuss:** were the models any good? What other "clues" (columns) might
help predict profit?